# 02 · EDA & Data Cleaning
**FTTH Rollout Optimizer — WBS Capstone**

Goals:
- Understand distributions, outliers, correlations
- Identify and handle missing values
- Flag and treat outliers
- Output: `data/processed/gemeinden_clean.csv`

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

PROCESSED = Path('../data/processed')
df = pd.read_csv(PROCESSED / 'gemeinden_raw.csv')

sns.set_theme(style='whitegrid', palette='Blues_d')
plt.rcParams['figure.dpi'] = 120

print(f'Loaded: {df.shape[0]} rows × {df.shape[1]} columns')
df.head(3)

## 1 · Basic Inspection

In [ ]:
print('=== dtypes ===')
print(df.dtypes)
print('\n=== Missing values ===')
missing = df.isnull().sum()
print(missing[missing > 0] if missing.sum() > 0 else 'No missing values')
print('\n=== Duplicates ===')
print(f'Duplicate rows: {df.duplicated().sum()}')

In [ ]:
df.describe().T[['mean','std','min','25%','50%','75%','max']].round(2)

## 2 · Target Variable Distribution

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4))

# Histogram
axes[0].hist(df['adoption_rate_pct'], bins=30, color='#378ADD', edgecolor='white', linewidth=0.5)
axes[0].set_title('Adoption Rate Distribution')
axes[0].set_xlabel('Adoption Rate (%)')
axes[0].set_ylabel('Count')
axes[0].axvline(df['adoption_rate_pct'].mean(), color='#E24B4A', linestyle='--', label=f"Mean: {df['adoption_rate_pct'].mean():.1f}%")
axes[0].axvline(df['adoption_rate_pct'].median(), color='#3B6D11', linestyle='--', label=f"Median: {df['adoption_rate_pct'].median():.1f}%")
axes[0].legend(fontsize=9)

# Boxplot
axes[1].boxplot(df['adoption_rate_pct'], patch_artist=True,
                boxprops=dict(facecolor='#B5D4F4', color='#185FA5'),
                medianprops=dict(color='#185FA5', linewidth=2))
axes[1].set_title('Adoption Rate Boxplot')
axes[1].set_ylabel('Adoption Rate (%)')
axes[1].set_xticks([])

# Priority tier breakdown
bins   = [0, 30, 50, 65, 100]
labels = ['Low (<30%)', 'Medium (30-50%)', 'High (50-65%)', 'Very High (>65%)']
colors = ['#E24B4A', '#BA7517', '#378ADD', '#3B6D11']
df['priority_tier'] = pd.cut(df['adoption_rate_pct'], bins=bins, labels=labels)
counts = df['priority_tier'].value_counts().reindex(labels)
axes[2].bar(range(len(labels)), counts.values, color=colors, edgecolor='white')
axes[2].set_xticks(range(len(labels)))
axes[2].set_xticklabels(labels, rotation=20, ha='right', fontsize=8)
axes[2].set_title('Priority Tier Breakdown')
axes[2].set_ylabel('Number of Gemeinden')

plt.tight_layout()
plt.savefig(PROCESSED / 'eda_target_distribution.png', bbox_inches='tight')
plt.show()
print(f"Tier counts:\n{counts.to_string()}")

## 3 · Feature Distributions

In [ ]:
FEATURES = [
    'population', 'pop_density_km2', 'avg_household_income',
    'existing_coverage_pct', 'dist_to_pop_node_m', 'dist_to_cabinet_m',
    'building_density', 'terrain_slope_deg', 'competitor_coverage_pct',
    'share_over_65_pct', 'unemployment_rate_pct', 'road_length_km', 'homes_passed'
]

fig, axes = plt.subplots(4, 4, figsize=(16, 12))
axes = axes.flatten()

for i, col in enumerate(FEATURES):
    axes[i].hist(df[col], bins=25, color='#85B7EB', edgecolor='white', linewidth=0.4)
    axes[i].set_title(col.replace('_', ' '), fontsize=9)
    axes[i].set_xlabel('')
    skew = df[col].skew()
    axes[i].text(0.97, 0.92, f'skew={skew:.2f}', transform=axes[i].transAxes,
                 fontsize=7, ha='right', color='#5F5E5A')

# hide unused subplots
for j in range(len(FEATURES), len(axes)):
    axes[j].set_visible(False)

plt.suptitle('Feature Distributions', fontsize=13, y=1.01)
plt.tight_layout()
plt.savefig(PROCESSED / 'eda_feature_distributions.png', bbox_inches='tight')
plt.show()

## 4 · Correlation Heatmap

In [ ]:
corr_cols = FEATURES + ['adoption_rate_pct']
corr = df[corr_cols].corr()

fig, ax = plt.subplots(figsize=(13, 10))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(
    corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
    center=0, vmin=-1, vmax=1,
    linewidths=0.5, linecolor='white',
    annot_kws={'size': 7}, ax=ax
)
ax.set_title('Feature Correlation Matrix', fontsize=13, pad=12)
plt.tight_layout()
plt.savefig(PROCESSED / 'eda_correlation_heatmap.png', bbox_inches='tight')
plt.show()

# Top correlations with target
target_corr = corr['adoption_rate_pct'].drop('adoption_rate_pct').sort_values(key=abs, ascending=False)
print('\nTop correlations with adoption_rate_pct:')
print(target_corr.to_string())

## 5 · Outlier Detection (IQR method)

In [ ]:
def flag_outliers_iqr(df, cols, factor=3.0):
    """Flag extreme outliers using IQR × factor. factor=3 = very conservative."""
    report = []
    for col in cols:
        Q1, Q3 = df[col].quantile(0.25), df[col].quantile(0.75)
        IQR = Q3 - Q1
        lo, hi = Q1 - factor * IQR, Q3 + factor * IQR
        n_out = ((df[col] < lo) | (df[col] > hi)).sum()
        report.append({'feature': col, 'lower_fence': round(lo,2),
                       'upper_fence': round(hi,2), 'n_outliers': n_out})
    return pd.DataFrame(report)

outlier_report = flag_outliers_iqr(df, FEATURES)
print('Outlier report (IQR × 3.0):')
print(outlier_report[outlier_report['n_outliers'] > 0].to_string(index=False))

In [ ]:
# Winsorise only columns with outliers — cap at 1st/99th percentile
cols_to_winsorise = ['population', 'dist_to_pop_node_m', 'pop_density_km2', 'homes_passed']

df_clean = df.copy()
for col in cols_to_winsorise:
    if col in df_clean.columns:
        p01 = df_clean[col].quantile(0.01)
        p99 = df_clean[col].quantile(0.99)
        df_clean[col] = df_clean[col].clip(lower=p01, upper=p99)
        print(f'  {col}: clipped to [{p01:.1f}, {p99:.1f}]')

print(f'\nShape after cleaning: {df_clean.shape}')

## 6 · Scatter Plots — Key Feature vs Target

In [ ]:
top_features = target_corr.abs().nlargest(6).index.tolist()

fig, axes = plt.subplots(2, 3, figsize=(14, 8))
axes = axes.flatten()

for i, feat in enumerate(top_features):
    axes[i].scatter(df_clean[feat], df_clean['adoption_rate_pct'],
                    alpha=0.35, s=18, color='#378ADD', edgecolors='none')
    # Trend line
    z = np.polyfit(df_clean[feat], df_clean['adoption_rate_pct'], 1)
    p = np.poly1d(z)
    x_line = np.linspace(df_clean[feat].min(), df_clean[feat].max(), 100)
    axes[i].plot(x_line, p(x_line), color='#E24B4A', linewidth=1.5, linestyle='--')
    r = df_clean[[feat, 'adoption_rate_pct']].corr().iloc[0, 1]
    axes[i].set_title(f'{feat.replace("_", " ")} (r={r:.2f})', fontsize=9)
    axes[i].set_xlabel(feat.replace('_', ' '), fontsize=8)
    axes[i].set_ylabel('Adoption Rate (%)', fontsize=8)

plt.suptitle('Top Features vs Adoption Rate', fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig(PROCESSED / 'eda_scatter_top_features.png', bbox_inches='tight')
plt.show()

## 7 · Geographic Distribution

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Adoption rate by location
sc1 = axes[0].scatter(
    df_clean['lon'], df_clean['lat'],
    c=df_clean['adoption_rate_pct'],
    cmap='RdYlGn', s=25, alpha=0.7, edgecolors='none'
)
plt.colorbar(sc1, ax=axes[0], label='Adoption Rate (%)')
axes[0].set_title('Adoption Rate — Geographic Distribution')
axes[0].set_xlabel('Longitude')
axes[0].set_ylabel('Latitude')
axes[0].set_xlim(9.5, 14.2)
axes[0].set_ylim(47.0, 51.0)

# Population density by location
sc2 = axes[1].scatter(
    df_clean['lon'], df_clean['lat'],
    c=df_clean['pop_density_km2'],
    cmap='Blues', s=25, alpha=0.7, edgecolors='none'
)
plt.colorbar(sc2, ax=axes[1], label='Pop. Density (inh/km²)')
axes[1].set_title('Population Density — Geographic Distribution')
axes[1].set_xlabel('Longitude')
axes[1].set_ylabel('Latitude')
axes[1].set_xlim(9.5, 14.2)
axes[1].set_ylim(47.0, 51.0)

plt.tight_layout()
plt.savefig(PROCESSED / 'eda_geographic_distribution.png', bbox_inches='tight')
plt.show()

## 8 · Save Clean Dataset

In [ ]:
# Drop helper column before saving
df_clean = df_clean.drop(columns=['priority_tier'], errors='ignore')

out_path = PROCESSED / 'gemeinden_clean.csv'
df_clean.to_csv(out_path, index=False)

print(f'Clean dataset saved → {out_path}')
print(f'Shape: {df_clean.shape}')
print(f'\nColumns: {df_clean.columns.tolist()}')

---
## EDA Summary

| Finding | Action |
|---------|--------|
| No missing values | No imputation needed |
| `population`, `dist_to_pop_node_m` right-skewed | Winsorised at p01/p99 |
| Top correlated features with target | `pop_density_km2`, `avg_household_income`, `dist_to_pop_node_m`, `building_density` |
| Geographic clustering visible | Spatial features important — include in model |

**Next:** `03_feature_engineering.ipynb`